# Vector similarity-grouping lab

Extracts one page's raw `Vector`s (no clustering/classification), groups them by shape via
`Vector_Similarity.similarity.vector_similarity_group` (item-type-count signature -> PCA-align ->
unit-box scale -> origin-translate -> mean-squared point-cloud distance), then, sorted by
member count descending, renders each group as **representative on the left / every member
isolated together on the right** so the similarity threshold can be eyeballed and tuned before
it's wired into the real pipeline (see `pipelines/_steps.py::reclassify_by_similarity`).

## 0 - Config

In [ ]:
from pathlib import Path

PDF_PATH = None       # None -> first PDF under references/
PAGE_INDEX = 0
MSE_THRESHOLD = None  # None -> config.SIMILARITY_MSE_THRESHOLD
MAX_GROUPS_SHOWN = 30
RENDER_DPI = 150

## 1 - Path bootstrap

In [ ]:
import os, sys
_root = os.path.abspath(os.path.join('../..'))
if _root not in sys.path:
    sys.path.append(_root)

## 2 - Extract vectors

`Reader` + `extract_vectors(page)` -- raw `get_drawings()` geometry, no clustering, no
classification, no FAST.

In [ ]:
from rastervec.Reader.reader import Reader
from rastervec.Vector.vector import extract_vectors

if PDF_PATH is None:
    refs = sorted((Path(_root) / 'references').glob('*.pdf'))
    assert refs, 'no PDFs found under references/'
    PDF_PATH = str(refs[0])

with Reader(PDF_PATH) as reader:
    page = reader.get_page(PAGE_INDEX)
    vectors = extract_vectors(page)

print(PDF_PATH, 'page', PAGE_INDEX, '->', len(vectors), 'vectors')

## 3 - Group by similarity

In [ ]:
from rastervec import config as rvconfig
from rastervec.Vector_Similarity.similarity import vector_similarity_group

threshold = MSE_THRESHOLD if MSE_THRESHOLD is not None else rvconfig.SIMILARITY_MSE_THRESHOLD
groups = vector_similarity_group(vectors, mse_threshold=threshold)
groups.sort(key=lambda g: len(g.members), reverse=True)

print(f'{len(vectors)} vectors -> {len(groups)} similarity groups (threshold={threshold})')
for g in groups[:15]:
    print(f'  {len(g.members):4d} members  signature={g.signature}')

## 4 - Render: representative (left) vs. all members isolated (right)

One row per group (top groups by member count first). `render_vector_cluster` isolates a
cluster onto its own tight canvas -- the representative gets its own single-vector render, the
member panel renders every group member together (still just that group's own vectors, not
positioned on the page).

In [ ]:
import matplotlib.pyplot as plt

from rastervec.renderer import render_vector_cluster

shown = groups[:MAX_GROUPS_SHOWN]
fig, axes = plt.subplots(len(shown), 2, figsize=(6, 3 * len(shown)))
if len(shown) == 1:
    axes = [axes]

for row, g in zip(axes, shown):
    ax_rep, ax_all = row
    rep_img = render_vector_cluster([g.representative], RENDER_DPI)
    ax_rep.imshow(rep_img)
    ax_rep.set_title(f'representative (n={len(g.members)})')
    ax_rep.axis('off')

    try:
        all_img = render_vector_cluster(g.members, RENDER_DPI)
        ax_all.imshow(all_img)
    except Exception as exc:  # noqa: BLE001 -- a lab notebook, keep going on a bad group
        ax_all.text(0.5, 0.5, str(exc), ha='center', va='center', wrap=True)
    ax_all.set_title(f'all {len(g.members)} members, isolated together')
    ax_all.axis('off')

plt.tight_layout()
plt.show()